In [1]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

# Load data
train = pd.read_csv('home-data-for-ml-course/train.csv')
test = pd.read_csv('home-data-for-ml-course/test.csv')

In [2]:
# Separate target
y = train.SalePrice
X = train.drop(['SalePrice'], axis=1)

In [3]:
from sklearn.base import BaseEstimator, TransformerMixin

class FeatureCreator(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X = X.copy()
        X['TotalSF'] = X['1stFlrSF'] + X['2ndFlrSF']
        return X



In [4]:
numeric_cols = [cname for cname in X.columns if X[cname].dtype in ['int64', 'float64']] + ['TotalSF']
categorical_cols = [cname for cname in X.columns if X[cname].nunique() < 10 and X[cname].dtype == "object"]

In [5]:
# 5. Define Transformers and Preprocessor
numerical_transformer = SimpleImputer(strategy='median')
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ])


In [6]:
# 6. Define the full Pipeline
my_pipeline = Pipeline(steps=[
    ('feature_creator', FeatureCreator()), 
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(n_estimators=300, random_state=42))
])

In [7]:
# 7. Run!
scores = -1 * cross_val_score(my_pipeline, X, y, cv=5, scoring='neg_root_mean_squared_error')
print(f"Pipeline RMSE: {scores.mean():.4f}")

Pipeline RMSE: 30580.5719


In [ ]:
# 8. Train on full dataset and make predictions
my_pipeline.fit(X, y)
preds = my_pipeline.predict(test[X.columns])

In [8]:
submission = pd.DataFrame({'Id': test['Id'], 'SalePrice': preds})
submission.to_csv('submission.csv', index=False)
submission.head()

NameError: name 'preds' is not defined